# 01 Collecting the reviews

Goal: build a corpus of Google Play reviews for Zimbabwean mobile-money and digital-banking apps.

Do not re-run this notebook unless you want a new snapshot. Every other notebook uses `data/raw_reviews.csv`,
scraped on 25 October 2026. Scraping again gives a different set (new reviews appear, some are deleted), so the results
would no longer match the reported results. The outputs from the original run are written below each step.

Tool: `google-play-scraper` (no API key needed).

In [ ]:
from google_play_scraper import search, app, reviews, Sort

import time
import pandas as pd

## 1. Finding the app IDs

The scraper needs each app's Play Store ID (for example `com.ecocash`), not its name.

Finding: the library has a known bug where the top search result comes back with the ID `None` (Google shows that
result in a special "featured" box). We got those IDs from lower positions in other searches, and from the address bar
of the app's Play Store page (`play.google.com/store/apps/details?id=...`).

In [ ]:
queries = ["EcoCash", "Ecocash super app", "innbucks", "cbz touch", "O'mari", "My ZB app",
           "ONe money mobile", "CABS MObile Banking"]

for q in queries:
    results = search(q, lang="en", country="zw", n_hits=5)
    print(f"\n=== Results for '{q}' ===")
    for a in results:
        print(a["appId"], "|", a["title"], "|", a["score"])

## 2. The study design

| Split | Apps | Why |
|---|---|---|
| Train | EcoCash (older app), InnBucks, CBZ Touch | The largest apps; both mobile money and banking |
| Test: known brand | EcoCash Super App | A new app from a brand the model has seen. Reported separately because it shares vocabulary with the old EcoCash app |
| Test: unseen brands | O'mari, OneMoney, MyZB, CABS | Brands the model never sees in training: the real test of generalisation |

Both groups contain mobile-money and banking apps, so a weaker test score cannot simply be blamed on "banking language".

In [ ]:
APPS = {
    "EcoCash":           {"id": "com.ecocash",             "brand": "EcoCash",  "split": "train"},
    "InnBucks":          {"id": "com.innbucks.customer",   "brand": "InnBucks", "split": "train"},
    "CBZ Touch":         {"id": "com.mode.cbz.ui",         "brand": "CBZ",      "split": "train"},
    "EcoCash Super App": {"id": "com.ecocash.superapp",    "brand": "EcoCash",  "split": "test"},
    "O'mari":            {"id": "com.hypercubesoft.omari", "brand": "O'mari",   "split": "test"},
    "MyZB":              {"id": "PASTE_ID",                "brand": "ZB",       "split": "test"},
    "OneMoney":          {"id": "PASTE_ID",                "brand": "OneMoney", "split": "test"},
    "CABS":              {"id": "PASTE_ID",                "brand": "CABS",     "split": "test"},
}

for name, info in APPS.items():
    try:
        d = app(info["id"], lang="en", country="zw")
        print(f"{name:18} | {d['title']:25} | score {d['score']:.2f} | "
              f"{d['ratings']:>5} ratings | {d['reviews']:>5} reviews")
    except Exception as e:
        print(f"{name:18} | ERROR: {e}")

Recorded output (24 Oct 2026):

| App | Score | Ratings | Written reviews |
|---|---|---|---|
| EcoCash | 4.41 | 1,968 | 798 |
| EcoCash Super App | 3.93 | 3,400 | 1,199 |
| InnBucks | 4.25 | 6,118 | 1,577 |
| O'mari | 4.07 | 375 | 180 |
| CBZ Touch | 3.69 | 3,350 | 1,629 |
| MyZB | 2.95 | 713 | 359 |
| OneMoney | 3.58 | 231 | 74 |
| CABS | 3.68 | 727 | 402 |

The figure on the Play Store page ("1.97K reviews") is really the number of ratings. Only 25 to 50% of people who rate
an app also write something, a self-selection bias.

## 3. Does `lang="en"` hide Shona or Ndebele reviews? (an experiment)

Google filters reviews by language. We tested whether asking for other languages returns extra reviews. Kalanga and
Tonga have no two-letter language code, so their three-letter codes are used (`kck`, `toi`). Note that `to` would be
Tongan, a language from the Pacific.

In [ ]:
def scrape_app(app_id, lang, max_reviews=5000, batch_size=200):
    """Download an app's reviews, newest first, one page per second."""
    collected = []
    token = None
    while len(collected) < max_reviews:
        batch, token = reviews(app_id, lang=lang, country="zw", sort=Sort.NEWEST,
                               count=batch_size, continuation_token=token)
        if not batch:
            break
        collected.extend(batch)
        time.sleep(1)
    return collected

LANGUAGES = {"English": "en", "Shona": "sn", "Ndebele": "nd", "Zulu": "zu", "Kalanga": "kck", "Tonga": "toi"}

seen = {}
for lang_name, code in LANGUAGES.items():
    try:
        result = scrape_app(APPS["EcoCash Super App"]["id"], code, max_reviews=2000)
        seen[lang_name] = {r["reviewId"] for r in result}
        print(f"{lang_name:8} ({code:3}): {len(result)} reviews")
    except Exception as e:
        print(f"{lang_name:8} ({code:3}): ERROR - {e}")

english_ids = seen.get("English", set())
for lang_name, ids in seen.items():
    if lang_name != "English":
        print(f"{lang_name:8}: {len(ids - english_ids)} reviews not found in English")

Recorded output: English, Shona, Ndebele, Kalanga and Tonga each returned the same 1,195 reviews. Zulu returned 0.

* Zulu is a language Google supports, so it really filtered and found no Zulu-labelled reviews. That shows the filter
  exists.
* The other codes are not supported, so Google fell back to its default and returned the English set again (0 new
  reviews).
* 1,195 of the 1,199 reported reviews came back from the English request.

Conclusion: one English scrape gets almost every written review. Shona and Ndebele reviews are already inside it, not
hidden from it, so mixed-language text is dealt with when modelling, not when collecting.

## 4. Scraping all eight apps

In [ ]:
# English only (see section 3)
LANGUAGES = {"English": "en"}

all_reviews = []
for app_name, info in APPS.items():
    print(f"Scraping {app_name}...", end=" ")
    try:
        result = scrape_app(info["id"], "en")
    except Exception as e:
        print(f"ERROR - {e}")
        continue
    for r in result:
        r["app"], r["brand"], r["split"] = app_name, info["brand"], info["split"]
    all_reviews.extend(result)
    print(f"{len(result)} reviews")
print(f"\nTotal: {len(all_reviews)} reviews")

Recorded output: EcoCash 795, InnBucks 1,576, CBZ Touch 1,628, EcoCash Super App 1,195, O'mari 180, MyZB 358,
OneMoney 74, CABS 402. Total 6,208 reviews, over 99.5% of the written reviews of every app.

## 5. Ethics step and saving

* Only the columns the analysis needs are kept. `userName` and `userImage` (personal data) and the developers' replies
  are dropped.
* Personal data inside the review text (phone numbers, e-mails) is masked during cleaning (rule 10 in
  `src/absa/text.py`).
* The data stays private: `raw_reviews.csv` must not be published, for example on a public GitHub repository.

In [ ]:
import os

df = pd.DataFrame(all_reviews)

# drop user fields
KEEP = ["reviewId",
        "app", "brand", "split",
        "content",
        "score",
        "at",
        "thumbsUpCount",
        "appVersion"]
df = df[KEEP]

before = len(df)
df = df.drop_duplicates(subset="reviewId")
print(f"Removed {before - len(df)} duplicates. {len(df)} reviews remain.")   # recorded: 0 removed

os.makedirs("data", exist_ok=True)
df.to_csv("data/raw_reviews.csv", index=False, encoding="utf-8-sig")